# 12 - Model Comparison

Compares multiple regression families on the laptop price dataset.
All evaluated on the same time-based split (train 2018-2023, test 2024+).

Models:
- Ridge (linear baseline)
- ElasticNet
- RandomForest
- HistGradientBoosting (current champion)
- LightGBM (if installed)
- CatBoost (if installed)

In [ ]:
# --- project bootstrap ---
import os, sys, pathlib
_here = pathlib.Path.cwd()
for _cand in (_here, *_here.parents):
    if (_cand / "pyproject.toml").exists():
        PROJECT_ROOT = _cand
        break
else:
    raise RuntimeError("pyproject.toml not found")
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"project root: {PROJECT_ROOT}")

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import time
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

from laptop_price.data import load_model_ready
from laptop_price.features.build import TARGET, NUMERIC_FEATURES, CATEGORICAL_FEATURES
from laptop_price.evaluation.splits import time_based_split
from laptop_price.evaluation.metrics import regression_metrics
from laptop_price.models.pipeline import build_pipeline

df = load_model_ready()
print(f"{len(df):,} listings")

## 1. Train / test split

Time-based: train 2018-2023, test 2024+.

In [ ]:
split = time_based_split(df["listing_year"], train_max_year=2023)
X_train = df.iloc[split.train_idx][[*NUMERIC_FEATURES, *CATEGORICAL_FEATURES]]
y_train = df.iloc[split.train_idx][TARGET]
X_test  = df.iloc[split.test_idx][[*NUMERIC_FEATURES, *CATEGORICAL_FEATURES]]
y_test  = df.iloc[split.test_idx][TARGET]
print(f"Train: {len(X_train):,}  Test: {len(X_test):,}")

## 2. Model definitions

In [ ]:
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer


def _cat_ordinal():
    return Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="UNKNOWN")),
                     ("enc", OrdinalEncoder(handle_unknown="use_encoded_value",
                                           unknown_value=-1))])


def _preprocessor(scale=False):
    num_steps = [("imp", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("sc", StandardScaler()))
    return ColumnTransformer(
        [("num", Pipeline(num_steps), list(NUMERIC_FEATURES)),
         ("cat", _cat_ordinal(), list(CATEGORICAL_FEATURES))],
        remainder="drop"
    )


model_factories = {
    "Ridge (baseline)": lambda: Pipeline([("pre", _preprocessor(scale=True)),
                                          ("model", Ridge(alpha=10.0))]),
    "ElasticNet":       lambda: Pipeline([("pre", _preprocessor(scale=True)),
                                          ("model", ElasticNet(alpha=0.1, l1_ratio=0.5,
                                                               max_iter=2000))]),
    "RandomForest":     lambda: Pipeline([("pre", _preprocessor()),
                                          ("model", RandomForestRegressor(
                                              n_estimators=200, max_depth=12,
                                              n_jobs=-1, random_state=42))]),
    "HistGBR (current)": lambda: build_pipeline(),
}

# Optional: LightGBM
try:
    import lightgbm as lgb
    model_factories["LightGBM"] = lambda: Pipeline([
        ("pre", _preprocessor()),
        ("model", lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05,
                                     num_leaves=63, random_state=42, verbose=-1))
    ])
except ImportError:
    print("LightGBM not installed; skipping.")

# Optional: CatBoost
try:
    from laptop_price.models.pipeline import build_catboost_pipeline
    model_factories["CatBoost"] = build_catboost_pipeline
except Exception:
    print("CatBoost not available; skipping.")

print(f"Models to compare: {list(model_factories)}")

## 3. Evaluation

Same feature set, same split, same metric for all models.

In [ ]:
results = []

for name, factory in model_factories.items():
    pipe = factory()
    t0 = time.perf_counter()
    pipe.fit(X_train, y_train)
    train_s = time.perf_counter() - t0

    t1 = time.perf_counter()
    y_pred = pipe.predict(X_test)
    infer_ms = (time.perf_counter() - t1) / len(X_test) * 1000

    m = regression_metrics(y_test, y_pred, label=name)
    m["train_s"]        = round(train_s, 2)
    m["infer_ms_per_k"] = round(infer_ms, 3)
    results.append(m)
    print(f"{name:25s}: R2={m['R2']:.4f}  MedAPE={m['MedAPE']:.2f}%  MAE={m['MAE']:,.0f}  train={train_s:.1f}s")

cmp = pd.DataFrame(results)
display(cmp[["label", "R2", "MAE", "MedAPE", "within_20pct", "train_s", "infer_ms_per_k"]])

## 4. Visualisation

In [ ]:
fig = px.bar(
    cmp.sort_values("MedAPE"),
    x="label",
    y=["R2", "MedAPE"],
    barmode="group",
    title="Model comparison: R2 and MedAPE (lower MedAPE is better)",
    labels={"label": "Model", "value": "Score"},
)
fig.show()

# Training time vs accuracy scatter
fig2 = px.scatter(
    cmp,
    x="train_s",
    y="MedAPE",
    text="label",
    title="Training time vs MedAPE (lower is better on both axes)",
    labels={"train_s": "Training time (s)", "MedAPE": "Median APE (%)"},
)
fig2.update_traces(textposition="top center")
fig2.show()

## Summary

- HistGBR and LightGBM typically dominate on tree-friendly tabular data
- Ridge provides a linear baseline and trains in milliseconds
- CatBoost handles categoricals natively but uses a separate preprocessor
- Ensemble: stacking HistGBR + LightGBM + Ridge as meta-learner is left as an exercise